In [1]:
# 셀1 - 데이터 로드  
import os, numpy as np, time
os.chdir('/home/xilinx/jupyter_notebooks/mobilenet')
# 1) 데이터 로드
W_CONV_LAYER = 1525656
B_CONV_LAYER = 17056
FC_LAYER     = 360000
B_FC_LAYER   = 1000
IMAGE_SIZE   = 224*224*3

def load_dat(fp):
    t0 = time.time()
    with open(fp,'r') as f: a = np.fromstring(f.read(), sep=' ',
dtype=np.int32)
    print(f"  {fp}: {len(a):,} ints in {time.time()-t0:.1f}s")
    return a

image_data   = load_dat('image_int.dat')[:IMAGE_SIZE]
all_w        = load_dat('weights.dat')
weights_CONV = all_w[:W_CONV_LAYER]
weights_FC   = all_w[W_CONV_LAYER:W_CONV_LAYER+FC_LAYER]
all_b        = load_dat('bias.dat')
bias_CONV    = all_b[:B_CONV_LAYER]
bias_FC      = all_b[B_CONV_LAYER:B_CONV_LAYER+B_FC_LAYER]

tile_3x3   = np.fromfile('tile_3x3.bin',   dtype=np.int32)
tile_convs = np.fromfile('tile_convs.bin', dtype=np.int32)
tile_avg   = np.fromfile('tile_avg.bin',   dtype=np.int32)
tile_fc    = np.fromfile('tile_fc.bin',    dtype=np.int32)
info_3x3   = np.fromfile('info_3x3.bin',   dtype=np.int32)
info_convs = np.fromfile('info_convs.bin', dtype=np.int32)
info_avg   = np.fromfile('info_avg.bin',   dtype=np.int32)
info_fc    = np.fromfile('info_fc.bin',    dtype=np.int32)

print()
print(f"image      : {len(image_data):>10,} ints")
print(f"w_conv     : {len(weights_CONV):>10,} ints")
print(f"w_fc       : {len(weights_FC):>10,} ints")
print(f"b_conv     : {len(bias_CONV):>10,} ints")
print(f"b_fc       : {len(bias_FC):>10,} ints")
print(f"tile_3x3   : {len(tile_3x3):>10,} ints")
print(f"tile_convs : {len(tile_convs):>10,} ints")
print(f"tile_avg   : {len(tile_avg):>10,} ints")
print(f"tile_fc    : {len(tile_fc):>10,} ints")
print(f"info_3x3   : {len(info_3x3):>10,} ints")
print(f"info_convs : {len(info_convs):>10,} ints")
print(f"info_avg   : {len(info_avg):>10,} ints")
print(f"info_fc    : {len(info_fc):>10,} ints")


  image_int.dat: 150,528 ints in 0.1s
  weights.dat: 1,885,656 ints in 1.1s
  bias.dat: 57,056 ints in 0.0s

image      :    150,528 ints
w_conv     :  1,525,656 ints
w_fc       :    360,000 ints
b_conv     :     17,056 ints
b_fc       :      1,000 ints
tile_3x3   :        768 ints
tile_convs :     64,800 ints
tile_avg   :        120 ints
tile_fc    :      1,920 ints
info_3x3   :      4,864 ints
info_convs :    410,400 ints
info_avg   :        760 ints
info_fc    :     12,160 ints


In [2]:
# 셀 2 - overlay  
from pynq import Overlay, MMIO, allocate
# 2) overlay + ip 확인
ol = Overlay('mobilenet.bit')
print("Bitstream loaded.")
print("IPs:", list(ol.ip_dict.keys()))

  # CTRL_BUS 살아있는지 확인 (ap_idle = 1)
ctrl = MMIO(0x40000000, 0x10000)
ap = ctrl.read(0x00)
print(f"\nap_ctrl reg = 0x{ap:08x}  →  ap_idle = {(ap>>2)&1}")


Bitstream loaded.
IPs: ['MobileNet_Stream_0', 'axi_gpio_0', 'axi_dma_0', 'axi_dma_1', 'axi_dma_2', 'processing_system7_0']

ap_ctrl reg = 0x00000004  →  ap_idle = 1


In [3]:
 # 3) DDR 버퍼 할당 + 데이터 복사
# 셀 3 - 12 buffers
def alloc_and_copy(src):
    buf = allocate(shape=src.shape, dtype=np.int32)
    np.copyto(buf, src)
    buf.flush()
    return buf

t0 = time.time()
buf_w_conv     = alloc_and_copy(weights_CONV)
buf_w_fc       = alloc_and_copy(weights_FC)
buf_b_conv     = alloc_and_copy(bias_CONV)
buf_b_fc       = alloc_and_copy(bias_FC)
buf_tile_3x3   = alloc_and_copy(tile_3x3)
buf_tile_convs = alloc_and_copy(tile_convs)
buf_tile_avg   = alloc_and_copy(tile_avg)
buf_tile_fc    = alloc_and_copy(tile_fc)
buf_info_3x3   = alloc_and_copy(info_3x3)
buf_info_convs = alloc_and_copy(info_convs)
buf_info_avg   = alloc_and_copy(info_avg)
buf_info_fc    = alloc_and_copy(info_fc)
print(f"Allocated 12 buffers in {time.time()-t0:.1f}s")


Allocated 12 buffers in 0.1s


In [4]:
print('w_conv', hex(buf_w_conv.physical_address), buf_w_conv.nbytes)
print('w_fc', hex(buf_w_fc.physical_address), buf_w_fc.nbytes)
print('b_conv', hex(buf_b_conv.physical_address), buf_b_conv.nbytes)
print('b_fc', hex(buf_b_fc.physical_address), buf_b_fc.nbytes)
print('tile_3x3', hex(buf_tile_3x3.physical_address), buf_tile_3x3.nbytes)
print('tile_convs', hex(buf_tile_convs.physical_address),buf_tile_convs.nbytes)
print('tile_avg', hex(buf_tile_avg.physical_address), buf_tile_avg.nbytes)
print('tile_fc', hex(buf_tile_fc.physical_address), buf_tile_fc.nbytes)
print('info_3x3', hex(buf_info_3x3.physical_address), buf_info_3x3.nbytes)
print('info_convs', hex(buf_info_convs.physical_address),buf_info_convs.nbytes)
print('info_avg', hex(buf_info_avg.physical_address), buf_info_avg.nbytes)
print('info_fc', hex(buf_info_fc.physical_address), buf_info_fc.nbytes)


w_conv 0x15900000 6102624
w_fc 0x15f00000 1440000
b_conv 0x15860000 68224
b_fc 0x1584a000 4000
tile_3x3 0x1584c000 3072
tile_convs 0x15880000 259200
tile_avg 0x1584e000 480
tile_fc 0x15850000 7680
info_3x3 0x15858000 19456
info_convs 0x16100000 1641600
info_avg 0x15852000 3040
info_fc 0x158c0000 48640


In [5]:
# 4) 고정 포인터 4개 등록 (w_conv, b_conv, w_fc, b_fc)
# 셀 4 - 4 pointers
# 4-1) 포인터 쓰기
data_mmio = MMIO(0x40010000, 0x10000)
data_mmio.write(0x10, buf_w_conv.physical_address & 0xFFFFFFFF)
data_mmio.write(0x14, (buf_w_conv.physical_address >> 32) & 0xFFFFFFFF)
data_mmio.write(0x1C, buf_b_conv.physical_address & 0xFFFFFFFF)
data_mmio.write(0x20, (buf_b_conv.physical_address >> 32) & 0xFFFFFFFF)
data_mmio.write(0x28, buf_w_fc.physical_address & 0xFFFFFFFF)
data_mmio.write(0x2C, (buf_w_fc.physical_address >> 32) & 0xFFFFFFFF)
data_mmio.write(0x34, buf_b_fc.physical_address & 0xFFFFFFFF)
data_mmio.write(0x38, (buf_b_fc.physical_address >> 32) & 0xFFFFFFFF)
print('4 pointers written.')

4 pointers written.


In [6]:
# 4-2) 검증 (read back)
print('w_conv lo', hex(data_mmio.read(0x10)), 'hi',
hex(data_mmio.read(0x14)))
print('b_conv lo', hex(data_mmio.read(0x1C)), 'hi',
hex(data_mmio.read(0x20)))
print('w_fc   lo', hex(data_mmio.read(0x28)), 'hi',
hex(data_mmio.read(0x2C)))
print('b_fc   lo', hex(data_mmio.read(0x34)), 'hi',
hex(data_mmio.read(0x38)))


w_conv lo 0x15900000 hi 0x0
b_conv lo 0x15860000 hi 0x0
w_fc   lo 0x15f00000 hi 0x0
b_fc   lo 0x1584a000 hi 0x0


In [7]:
# pynq에서 dma 객체 잡기
# 셀 5 - DMA channels
ip_in  = ol.axi_dma_0.sendchannel
ip_out = ol.axi_dma_1.recvchannel
res_rd = ol.axi_dma_2.sendchannel
res_wr = ol.axi_dma_2.recvchannel
print('DMA channels OK:', ip_in, ip_out, res_rd, res_wr)


DMA channels OK: <pynq.lib.dma._SDMAChannel object at 0xb3887130> <pynq.lib.dma._SDMAChannel object at 0xb38e7e50> <pynq.lib.dma._SDMAChannel object at 0xac25b7c0> <pynq.lib.dma._SDMAChannel object at 0xac25b6e8>


In [8]:
 # 셀 6 - helpers
%run -i /home/xilinx/jupyter_notebooks/mobilenet/python/helpers.py
%run -i /home/xilinx/jupyter_notebooks/mobilenet/python/helpers_v2.py
%run -i /home/xilinx/jupyter_notebooks/mobilenet/python/helpers_v3.py

helpers.py loaded.
  ctrl @ 0x40000000, data @ 0x40010000
  ap_idle = 1
helpers_v2.py loaded.
  Functions: dram_2_stream_3x3_type2, dram_2_stream_1x1_exp0/1,
             stream_2_dram_3x3_exp0/1, stream_2_dram_1x1_exp0,
             tile/info offset helpers, conv_batch_relu_pe
helpers_v3.py loaded (SDK-correct port).
  dram_2_stream_3x3_v3, dram_2_stream_1x1_v3, dram_2_stream_array_v3
  stream_2_dram_3x3_v3, stream_2_dram_1x1_v3, stream_2_dram_array_v3
  call_ip_pe (universal IP call helper)


In [9]:
  # 셀 7 - run_layer0
%run -i /home/xilinx/jupyter_notebooks/mobilenet/python/run_layer0.py

=== Packing input (this may take 30-60s in pure Python) ===
  packed input: 1,382,400 ints  (8.9s)
  buf_input phys = 0x16300000
  per-PE output buffer: 100,352 ints x 4 PE

=== Calling IP per PE ===
  ap_idle before = 1
  PE 0: tile=0x1584c000, info=0x15858000 ... done in 114.3 ms
  PE 1: tile=0x1584c300, info=0x15859300 ... done in 114.2 ms
  PE 2: tile=0x1584c600, info=0x1585a600 ... done in 114.2 ms
  PE 3: tile=0x1584c900, info=0x1585b900 ... done in 114.3 ms

=== Reassembling output ===
  PE 0: wrote 100,352 ints
  PE 1: wrote 100,352 ints
  PE 2: wrote 100,352 ints
  PE 3: wrote 100,352 ints

=== Output sanity ===
  shape: (401408,)
  min/max: 0 / 1229027
  nonzero ratio: 57.8%
  first 8 values: [0 0 0 0 0 0 0 0]


In [10]:
  # 셀 9 - dram_2_stream_1x1_exp0_v2
  def dram_2_stream_1x1_exp0_v2(in_mem, depth_out, depth_in, length, stride):
      out = []
      push = out.append
      k_pos = 0
      for k in range(0, length, TILE_MAP):
          min_k = min(length, k + TILE_MAP); limit_k = min_k - k
          l_pos = 0
          for l in range(0, length, TILE_MAP):
              min_l = min(length, l + TILE_MAP); limit_l = min_l - l
              for i in range(0, depth_out // NUMBER_PE, TILE_CONV_OUT):
                  for PE in range(NUMBER_PE):
                      pe_start = PE * depth_in // NUMBER_PE
                      pe_end   = (PE + 1) * depth_in // NUMBER_PE
                      j_pos = 0
                      for j in range(pe_start, pe_end, TILE_CONV_IN):
                          min_j = min(pe_end, j + TILE_CONV_IN); limit_j = min_j - j
                          reg = (l_pos*limit_j + k_pos*limit_j + j_pos +
                                 PE*(length//stride)*(length//stride)*depth_in//NUMBER_PE)
                          n_xfer = limit_l*limit_k*limit_j//(stride*stride)
                          for x in range(reg, reg + n_xfer):
                              push(int(in_mem[x]))
                          j_pos += (length//stride)*(length//stride)*limit_j
              l_pos += limit_l*limit_k//(stride*stride)
          k_pos += (length//stride)*limit_k//stride
      return np.array(out, dtype=np.int32)

  print("v2 defined")


v2 defined


In [11]:
# 셀 10 - cpu_map 할당 + layer 0 결과 복사
CPU_MAP_SIZE = 1572864
buf_cpu_map = allocate((CPU_MAP_SIZE,), dtype=np.int32)
buf_cpu_map[:] = 0                     
buf_cpu_map[:401408] = out_layer0
buf_cpu_map.flush()
print(f"cpu_map [:8]:", np.asarray(buf_cpu_map)[:8])
print(f"cpu_map [700000:700008]:", np.asarray(buf_cpu_map)[700000:700008])


cpu_map [:8]: [0 0 0 0 0 0 0 0]
cpu_map [700000:700008]: [0 0 0 0 0 0 0 0]


In [12]:
  # 셀 11 - layer 1 (G1+G2)
buf_cpu_map[:401408] = out_layer0
buf_cpu_map.flush()

  # Layer 1 G1
buf_l1g1_outputs = []
for pe in range(NUMBER_PE):
    p = dram_2_stream_3x3_type2(np.asarray(buf_cpu_map), 32, 112, pe)
    bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
    bo = allocate((100352,), dtype=np.int32)
    conv_batch_relu_pe(1, 0, 2, pe, bi, bo, ip_in, ip_out, timeout=10)
    buf_l1g1_outputs.append(bo); del bi
stream_2_dram_3x3_exp0(buf_cpu_map, buf_l1g1_outputs, 32, 112, 1) 
  buf_cpu_map.flush()

  # Layer 1 G2
p = dram_2_stream_1x1_exp0_v2(np.asarray(buf_cpu_map), 16, 32, 112, 1)
bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
buf_l1g2_outputs = []
for pe in range(NUMBER_PE):
    bo = allocate((50176,), dtype=np.int32)
    conv_batch_relu_pe(1, 1, 1, pe, bi, bo, ip_in, ip_out, timeout=10)
    buf_l1g2_outputs.append(bo)
stream_2_dram_1x1_exp0(buf_cpu_map, buf_l1g2_outputs, 16, 112)  
buf_cpu_map.flush()

print(f"L1 first 8: {np.asarray(buf_cpu_map)[:8]}")

L1 first 8: [-42012    323 -16049 -19251 -19259 -17695 -22197 -18343]


In [13]:
  ans = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L1.txt', dtype=np.int32)
  ours = np.asarray(buf_cpu_map)[:200704]
  print(f"L1: {(ans==ours).sum()}/{len(ans)}")
  print(f"cm[200704:200712]:", np.asarray(buf_cpu_map)[200704:200712]) 

L1: 200704/200704
cm[200704:200712]: [256 256 256 256 256 256 256 256]


In [14]:
p = dram_2_stream_1x1_exp0_v2(np.asarray(buf_cpu_map), 96, 16, 112, 1)
print("p2g1 max:", p.max(), "min:", p.min())  

bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
buf_l2g1_outputs = []
for pe in range(NUMBER_PE):
    bo = allocate((24*112*112,), dtype=np.int32)
    bo[:] = 0   
    conv_batch_relu_pe(2, 0, 1, pe, bi, bo, ip_in, ip_out, timeout=10)
    buf_l2g1_outputs.append(bo)

stream_2_dram_1x1_v3(buf_cpu_map, buf_l2g1_outputs, 96, 112)
buf_cpu_map.flush()

  # 즉시 검증
ans = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L2_G1.txt', dtype=np.int32)
ours = np.asarray(buf_cpu_map)[:96*112*112]
print(f"L2 G1: {(ans==ours).sum()}/{len(ans)}({(ans==ours).sum()/len(ans)*100:.2f}%)")
print(f"PE 0 raw max:", buf_l2g1_outputs[0].max())
print(f"ans max:", ans.max(), "ours max:", ours.max())


p2g1 max: 330511 min: -387342
L2 G1: 1204224/1204224(100.00%)
PE 0 raw max: 951706
ans max: 994424 ours max: 994424


In [15]:
  # Layer 2 G2
  buf_l2g2_outputs = []
  for pe in range(NUMBER_PE):
      p = dram_2_stream_3x3_type2(np.asarray(buf_cpu_map), 96, 112, pe)
      bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
      bo = allocate((24*56*56,), dtype=np.int32)
      bo[:] = 0
      conv_batch_relu_pe(2, 1, 2, pe, bi, bo, ip_in, ip_out, timeout=15)
      buf_l2g2_outputs.append(bo); del bi
  stream_2_dram_3x3_v3(buf_cpu_map, buf_l2g2_outputs, 96, 112, 2)
  buf_cpu_map.flush()

  # 즉시 검증
  ans = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L2_G2.txt', dtype=np.int32)
  ours = np.asarray(buf_cpu_map)[:96*56*56]
  match = (ans == ours).sum()
  print(f"L2 G2: {match}/{len(ans)} ({match/len(ans)*100:.2f}%)")
  print(f"ans max: {ans.max()}, ours max: {ours.max()}")


L2 G2: 301056/301056 (100.00%)
ans max: 343763, ours max: 343763


In [16]:

# ============================================================
# 셀 16: Helper 함수 정의 (L2 G3, L3 ~ 사용)
# ============================================================
import time, numpy as np
from pynq import allocate

def reset_all_dma():
    ip_in._mmio.write(0x00, 0x4); res_rd._mmio.write(0x00, 0x4)
    ip_out._mmio.write(0x30, 0x4); res_wr._mmio.write(0x30, 0x4)
    time.sleep(0.1)
    ip_in._mmio.write(0x00, 0x1); res_rd._mmio.write(0x00, 0x1)
    ip_out._mmio.write(0x30, 0x1); res_wr._mmio.write(0x30, 0x1)
    time.sleep(0.05)
    for ch in [ip_in, ip_out, res_rd, res_wr]:
        ch._first_transfer = True

def d2s_1x1_exp0(in_mem, depth_out, depth_in, length, stride):
    """testbench DRAM_2_STREAM_1x1 expansion=0 (PE-specific)"""
    in_arr = np.asarray(in_mem)
    out = []; push = out.append
    k_pos = 0
    for k in range(0, length, TILE_MAP):
        min_k = min(length, k + TILE_MAP); limit_k = min_k - k
        l_pos = 0
        for l in range(0, length, TILE_MAP):
            min_l = min(length, l + TILE_MAP); limit_l = min_l - l
            for i in range(0, depth_out // NUMBER_PE, TILE_CONV_OUT):
                for PE in range(NUMBER_PE):
                    j_pos = 0
                    for j in range(PE * depth_in // NUMBER_PE,
                                   (PE + 1) * depth_in // NUMBER_PE, TILE_CONV_IN):
                        min_j = min((PE + 1) * depth_in // NUMBER_PE, j + TILE_CONV_IN)
                        limit_j = min_j - j
                        reg = (l_pos*limit_j + k_pos*limit_j + j_pos
                               + PE*(length//stride)*(length//stride)*depth_in//NUMBER_PE)
                        n = limit_l*limit_k*limit_j//(stride*stride)
                        for x in range(reg, reg + n):
                            push(int(in_arr[x]))
                        j_pos += (length//stride)*(length//stride)*limit_j
            l_pos += limit_l*limit_k//(stride*stride)
        k_pos += (length//stride)*limit_k//stride
    return np.array(out, dtype=np.int32)

def s2d_1x1_exp1(out_mem, pe_list, depth, length, skip=0):
    """testbench STREAM_2_DRAM_1x1 expansion=1 (tile-based)"""
    multi = length*length
    dPE = depth // NUMBER_PE
    out_np = np.asarray(out_mem)
    exp = dPE * multi
    for PE in range(NUMBER_PE):
        pd = np.asarray(pe_list[PE])[skip:skip+exp]
        c = 0
        for j in range(0, length, TILE_MAP):
            mj = min(length, j + TILE_MAP)
            for k in range(0, length, TILE_MAP):
                mk = min(length, k + TILE_MAP)
                for i in range(PE*dPE, (PE+1)*dPE, TILE_CONV_OUT):
                    mi = min((PE+1)*dPE, i + TILE_CONV_OUT)
                    for l in range(i, mi):
                        px = l * multi
                        for m in range(j, mj):
                            py = m * length + px
                            for o in range(k, mk):
                                if c < len(pd): out_np[o+py] = pd[c]
                                c += 1

def s2d_1x1_exp0(out_mem, pe_list, depth, length, skip=0):
    """G3 output: expansion=0 sequential per PE"""
    out_np = np.asarray(out_mem)
    chunk = depth * length * length // NUMBER_PE
    pos = 0
    for PE in range(NUMBER_PE):
        pd = np.asarray(pe_list[PE])[skip:skip+chunk]
        out_np[pos:pos+len(pd)] = pd
        pos += chunk

def s2d_3x3_exp0(out_mem, pe_list, depth, length, stride, skip=0):
    """3x3 expansion=0: sequential per-PE"""
    out_np = np.asarray(out_mem)
    lo = length // stride
    chunk = depth * lo * lo // NUMBER_PE
    pos = 0
    for PE in range(NUMBER_PE):
        pd = np.asarray(pe_list[PE])[skip:skip+chunk]
        out_np[pos:pos+len(pd)] = pd
        pos += chunk

def s2d_3x3_exp1(out_mem, pe_list, depth, length, stride, skip=0):
    """testbench STREAM_2_DRAM_3x3 expansion=1"""
    lo = length // stride; multi = lo * lo
    dPE = depth // NUMBER_PE
    out_np = np.asarray(out_mem); exp = dPE * multi
    for PE in range(NUMBER_PE):
        pd = np.asarray(pe_list[PE])[skip:skip+exp]
        c = 0
        for i in range(PE*dPE, (PE+1)*dPE, TILE_CONV_OUT):
            mi = min((PE+1)*dPE, i + TILE_CONV_OUT)
            for j in range(0, length, TILE_MAP):
                mj = min(length, j + TILE_MAP) // stride; js = j // stride
                for k in range(0, length, TILE_MAP):
                    mk = min(length, k + TILE_MAP) // stride; ks = k // stride
                    for l in range(i, mi):
                        px = l * multi
                        for m in range(js, mj):
                            py = m * lo + px
                            for o in range(ks, mk):
                                if c < len(pd): out_np[o+py] = pd[c]
                                c += 1

def call_4pe(layer, inter, type_layer, in_buf, expected, label):
    """G1, G3 호출 (4 PE 같은 input broadcast)"""
    sz = expected + 1000
    outs = []
    for pe in range(NUMBER_PE):
        reset_all_dma()
        bo = allocate((sz,), dtype=np.int32); bo[:] = 0; bo.flush()
        rrecv = allocate((sz,), dtype=np.int32); rrecv[:] = 0; rrecv.flush()
        rbuf = allocate((100,), dtype=np.int32); rbuf[:] = 0; rbuf.flush()
        tp = buf_tile_convs.physical_address + tile_convs_offset_bytes(layer, inter, pe)
        ip_phys = buf_info_convs.physical_address + info_convs_offset_bytes(layer, inter, pe)
        ip_set_args(layer, inter, type_layer)
        ip_set_tile_info(tp, ip_phys)
        res_wr.transfer(rrecv); ip_out.transfer(bo)
        res_rd.transfer(rbuf); ip_in.transfer(in_buf)
        ctrl.write(0, 0x1)
        for _ in range(40):
            time.sleep(0.5)
            if (ctrl.read(0)>>1)&1: break
        nz = np.count_nonzero(np.asarray(bo))
        print(f"  {label} PE{pe}: bo[:4]={np.asarray(bo)[:4]} nz={nz}")
        outs.append(bo)
    return outs

def call_g2(layer, dmid, l_in, expected):
    """G2 호출 (PE별 dram_2_stream_3x3_type2 input)"""
    sz = expected + 1000
    outs = []
    for pe in range(NUMBER_PE):
        p = dram_2_stream_3x3_type2(np.asarray(buf_cpu_map), dmid, l_in, pe)
        bi_pe = allocate(p.shape, dtype=np.int32); bi_pe[:] = p; bi_pe.flush()
        reset_all_dma()
        bo = allocate((sz,), dtype=np.int32); bo[:] = 0; bo.flush()
        rrecv = allocate((sz,), dtype=np.int32); rrecv[:] = 0; rrecv.flush()
        rbuf = allocate((100,), dtype=np.int32); rbuf[:] = 0; rbuf.flush()
        tp = buf_tile_convs.physical_address + tile_convs_offset_bytes(layer, 1, pe)
        ip_phys = buf_info_convs.physical_address + info_convs_offset_bytes(layer, 1, pe)
        ip_set_args(layer, 1, 2)
        ip_set_tile_info(tp, ip_phys)
        res_wr.transfer(rrecv); ip_out.transfer(bo)
        res_rd.transfer(rbuf); ip_in.transfer(bi_pe)
        ctrl.write(0, 0x1)
        for _ in range(40):
            time.sleep(0.5)
            if (ctrl.read(0)>>1)&1: break
        nz = np.count_nonzero(np.asarray(bo))
        print(f"  G2 PE{pe}: bo[:4]={np.asarray(bo)[:4]} nz={nz}")
        outs.append(bo); del bi_pe
    return outs

# 어떤 skip이 정답인지 자동 시도
def best_layout(fn, out_mem, pe_list, *args):
    """skip=0 vs skip=2 둘 다 시도해서 더 잘 매치하는 쪽 선택"""
    DUMP = '/home/xilinx/jupyter_notebooks/mobilenet/dump'
    return None  # placeholder

print("Helpers defined ✓")


Helpers defined ✓


In [17]:

# ============================================================
# 셀 17: Layer 2 G3 (project, expansion=0 sequential)
# ============================================================
p = dram_2_stream_1x1_v3(np.asarray(buf_cpu_map), 24, 96, 56, 1, 1)
bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()

buf_l2g3 = []
for pe in range(NUMBER_PE):
    reset_all_dma()
    bo = allocate((20000,), dtype=np.int32); bo[:] = 0; bo.flush()
    rrecv = allocate((20000,), dtype=np.int32); rrecv[:] = 0; rrecv.flush()
    rbuf = allocate((100,), dtype=np.int32); rbuf[:] = 0; rbuf.flush()
    tp = buf_tile_convs.physical_address + tile_convs_offset_bytes(2, 2, pe)
    ip_phys = buf_info_convs.physical_address + info_convs_offset_bytes(2, 2, pe)
    ip_set_args(2, 2, 1)
    ip_set_tile_info(tp, ip_phys)
    res_wr.transfer(rrecv); ip_out.transfer(bo)
    res_rd.transfer(rbuf); ip_in.transfer(bi)
    ctrl.write(0, 0x1)
    for _ in range(15):
        time.sleep(0.5)
        if (ctrl.read(0)>>1)&1: break
    print(f"  L2G3 PE{pe}: bo[:4]={np.asarray(bo)[:4]} nz={np.count_nonzero(np.asarray(bo))}")
    buf_l2g3.append(bo)

# skip 0과 2 둘 다 시도
ans = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L2_G3.txt', dtype=np.int32)
best = (-1, 0)
for skip in [0, 2]:
    buf_cpu_map[:24*56*56] = 0; buf_cpu_map.flush()
    s2d_1x1_exp0(buf_cpu_map, buf_l2g3, 24, 56, skip=skip)
    buf_cpu_map.flush()
    m = (ans == np.asarray(buf_cpu_map)[:len(ans)]).sum()
    print(f"  skip={skip}: {m}/{len(ans)} ({100*m/len(ans):.2f}%)")
    if m > best[1]: best = (skip, m)

# 정답으로 다시 저장
buf_cpu_map[:24*56*56] = 0; buf_cpu_map.flush()
s2d_1x1_exp0(buf_cpu_map, buf_l2g3, 24, 56, skip=best[0])
buf_cpu_map.flush()
print(f"L2 G3 best: skip={best[0]}, match={best[1]}/{len(ans)} ({100*best[1]/len(ans):.2f}%)")

L2G3_SKIP = best[0]   # 다음 layer에서 같은 skip 사용


  L2G3 PE0: bo[:4]=[   3122 -125933 -122440 -127686] nz=18816
  L2G3 PE1: bo[:4]=[145930  57739  21780  26197] nz=18816
  L2G3 PE2: bo[:4]=[ -79286 -124595  -85584  -81792] nz=18816
  L2G3 PE3: bo[:4]=[ -6875 -97591 -77951 -72061] nz=18816
  skip=0: 75264/75264 (100.00%)
  skip=2: 1/75264 (0.00%)
L2 G3 best: skip=0, match=75264/75264 (100.00%)


In [18]:

# ============================================================
# 셀 18: Run InvertedResidual 함수 (L3 ~ L17 자동 사용)
# 매 단계 ans로 input 복구 + skip 자동 선택
# ============================================================
DUMP = '/home/xilinx/jupyter_notebooks/mobilenet/dump'

def restore_cpu_map(prev_layer_dump, expected_size):
    """이전 layer ans로 cpu_map 복구"""
    try:
        ans = np.loadtxt(f'{DUMP}/{prev_layer_dump}.txt', dtype=np.int32)
        buf_cpu_map[:] = 0
        buf_cpu_map[:len(ans)] = ans
        buf_cpu_map.flush()
        print(f"  restored {prev_layer_dump} ({len(ans)} ints)")
        return True
    except Exception as e:
        print(f"  ⚠️ {prev_layer_dump} 복구 실패: {e}")
        return False

def best_match(stream_fn, pe_outs, depth, length, ans, *extra_args):
    """skip 0/2 둘 다 시도. extra_args = stride 등"""
    buf_cpu_map[:depth*length*length // (extra_args[0]**2 if extra_args else 1)] = 0
    buf_cpu_map.flush()
    best_skip, best_m = 0, -1
    for skip in [0, 2]:
        if extra_args:
            stream_fn(buf_cpu_map, pe_outs, depth, length, *extra_args, skip=skip)
        else:
            stream_fn(buf_cpu_map, pe_outs, depth, length, skip=skip)
        buf_cpu_map.flush()
        m = (ans == np.asarray(buf_cpu_map)[:len(ans)]).sum()
        print(f"    skip={skip}: {100*m/len(ans):.2f}%")
        if m > best_m: best_skip, best_m = skip, m
        buf_cpu_map[:len(ans)] = 0; buf_cpu_map.flush()
    # 정답 skip으로 다시 저장
    if extra_args:
        stream_fn(buf_cpu_map, pe_outs, depth, length, *extra_args, skip=best_skip)
    else:
        stream_fn(buf_cpu_map, pe_outs, depth, length, skip=best_skip)
    buf_cpu_map.flush()
    return best_skip, best_m


def run_layer(layer, depth_in, depth_mid, depth_out, length_in, stride_g2, prev_dump):
    length_g2 = length_in // stride_g2
    exp = 0 if stride_g2 == 1 else 1
    print(f"\n{'='*60}")
    print(f"=== L{layer}: in={depth_in}@{length_in}², mid={depth_mid}, out={depth_out}, stride_g2={stride_g2}, exp={exp} ===")

    # cpu_map 복구
    if not restore_cpu_map(prev_dump, depth_in*length_in*length_in):
        return

    # ===== G1 (1x1 expand: depth_in→depth_mid, length stays) =====
    print(f"\n--- L{layer} G1 (1x1 expand) ---")
    p = d2s_1x1_exp0(buf_cpu_map, depth_mid, depth_in, length_in, 1)
    bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
    g1 = call_4pe(layer, 0, 1, bi, depth_mid//4 * length_in * length_in, "G1")
    ans_g1 = np.loadtxt(f'{DUMP}/L{layer}_G1.txt', dtype=np.int32)
    skip, m = best_match(s2d_1x1_exp1, g1, depth_mid, length_in, ans_g1)
    print(f"  L{layer} G1: skip={skip}, {m}/{len(ans_g1)} ({100*m/len(ans_g1):.2f}%)")
    # G2 input 복구
    buf_cpu_map[:len(ans_g1)] = ans_g1; buf_cpu_map.flush()

    # ===== G2 (3x3 depthwise) =====
    print(f"\n--- L{layer} G2 (3x3 depthwise) ---")
    g2 = call_g2(layer, depth_mid, length_in, depth_mid//4 * length_g2 * length_g2)
    ans_g2 = np.loadtxt(f'{DUMP}/L{layer}_G2.txt', dtype=np.int32)
    if exp == 0:
        skip, m = best_match(s2d_3x3_exp0, g2, depth_mid, length_in, ans_g2, stride_g2)
    else:
        skip, m = best_match(s2d_3x3_exp1, g2, depth_mid, length_in, ans_g2, stride_g2)
    print(f"  L{layer} G2: skip={skip}, {m}/{len(ans_g2)} ({100*m/len(ans_g2):.2f}%)")
    # G3 input 복구
    buf_cpu_map[:len(ans_g2)] = ans_g2; buf_cpu_map.flush()

    # ===== G3 (1x1 project) =====
    print(f"\n--- L{layer} G3 (1x1 project) ---")
    if exp == 0:
        p = d2s_1x1_exp0(buf_cpu_map, depth_out, depth_mid, length_g2, 1)
    else:
        p = dram_2_stream_1x1_v3(buf_cpu_map, depth_out, depth_mid, length_g2, 1, stride_g2)
    bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
    g3 = call_4pe(layer, 2, 1, bi, depth_out//4 * length_g2 * length_g2, "G3")
    ans_g3 = np.loadtxt(f'{DUMP}/L{layer}_G3.txt', dtype=np.int32)
    skip, m = best_match(s2d_1x1_exp0, g3, depth_out, length_g2, ans_g3)
    print(f"  L{layer} G3: skip={skip}, {m}/{len(ans_g3)} ({100*m/len(ans_g3):.2f}%)")

print("run_layer defined ✓")

run_layer defined ✓


In [19]:

# ============================================================
# 셀 19: L3 실행 (24→144→24, length 56, stride 1)
# ============================================================
run_layer(3, 24, 144, 24, 56, 1, 'L2_G3')



=== L3: in=24@56², mid=144, out=24, stride_g2=1, exp=0 ===
  restored L2_G3 (75264 ints)

--- L3 G1 (1x1 expand) ---
  G1 PE0: bo[:4]=[23867     0 26509 20426] nz=83675
  G1 PE1: bo[:4]=[    0 38013 43215 40189] nz=85269
  G1 PE2: bo[:4]=[166498 116297 102957 104014] nz=82522
  G1 PE3: bo[:4]=[     0  78918 122155 119316] nz=89888
    skip=0: 100.00%
    skip=2: 16.01%
  L3 G1: skip=0, 451584/451584 (100.00%)

--- L3 G2 (3x3 depthwise) ---
  G2 PE0: bo[:4]=[369764 302708 297452 268332] nz=74082
  G2 PE1: bo[:4]=[61685     0     0     0] nz=74580
  G2 PE2: bo[:4]=[     0 240527 237216 207931] nz=70644
  G2 PE3: bo[:4]=[56432     0     0     0] nz=70612
    skip=0: 100.00%
    skip=2: 24.40%
  L3 G2: skip=0, 451584/451584 (100.00%)

--- L3 G3 (1x1 project) ---
  G3 PE0: bo[:4]=[0 0 0 0] nz=0
  G3 PE1: bo[:4]=[0 0 0 0] nz=0
  G3 PE2: bo[:4]=[0 0 0 0] nz=0
  G3 PE3: bo[:4]=[0 0 0 0] nz=0
    skip=0: 0.00%
    skip=2: 0.00%
  L3 G3: skip=0, 0/75264 (0.00%)


In [20]:

  # === L3 G3 with residual add ===
  import time, numpy as np
  from pynq import allocate

  # L2 G3 ans (residual 데이터)
  ans_l2g3 = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L2_G3.txt', dtype=np.int32)

  # G3 input 준비 (cpu_map의 L3 G2 ans는 이미 있음)
  ans_l3g2 = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L3_G2.txt', dtype=np.int32)
  buf_cpu_map[:len(ans_l3g2)] = ans_l3g2; buf_cpu_map.flush()

  p = d2s_1x1_exp0(buf_cpu_map, 24, 144, 56, 1)
  bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()

  # residual data 준비 (L2 G3 ans를 PE 별로 분할)
  # 24 ch / 4 PE = 6 ch per PE
  chunk = 24 * 56 * 56 // 4  # 18816
  res_data = []
  for pe in range(4):
      rd = allocate((chunk + 100,), dtype=np.int32)
      rd[:chunk] = ans_l2g3[pe*chunk:(pe+1)*chunk]
      rd.flush()
      res_data.append(rd)

  g3 = []
  for pe in range(NUMBER_PE):
      reset_all_dma()
      bo = allocate((20000,), dtype=np.int32); bo[:] = 0; bo.flush()
      rrecv = allocate((20000,), dtype=np.int32); rrecv[:] = 0; rrecv.flush()

      tp = buf_tile_convs.physical_address + tile_convs_offset_bytes(3, 2, pe)
      ip_phys = buf_info_convs.physical_address + info_convs_offset_bytes(3, 2, pe)
      ip_set_args(3, 2, 1)
      ip_set_tile_info(tp, ip_phys)

      res_wr.transfer(rrecv); ip_out.transfer(bo)
      res_rd.transfer(res_data[pe])    # ⭐ PE별 residual data
      ip_in.transfer(bi)
      ctrl.write(0, 0x1)

      for _ in range(30):
          time.sleep(0.5)
          if (ctrl.read(0)>>1)&1: break
      nz = np.count_nonzero(np.asarray(bo))
      print(f"G3 PE{pe}: bo[:4]={np.asarray(bo)[:4]} nz={nz}")
      g3.append(bo)

  # 검증
  ans_l3g3 = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L3_G3.txt', dtype=np.int32)
  for skip in [0, 2]:
      buf_cpu_map[:24*56*56] = 0; buf_cpu_map.flush()
      s2d_1x1_exp0(buf_cpu_map, g3, 24, 56, skip=skip)
      buf_cpu_map.flush()
      m = (ans_l3g3 == np.asarray(buf_cpu_map)[:len(ans_l3g3)]).sum()
      print(f"L3 G3 skip={skip}: {m}/{len(ans_l3g3)} ({100*m/len(ans_l3g3):.2f}%)")


G3 PE0: bo[:4]=[ -13568 -101366  -94886 -227117] nz=18426
G3 PE1: bo[:4]=[  43921  157059   83533 -400537] nz=18816
G3 PE2: bo[:4]=[ 461759  768225 -264461 -249733] nz=18816
G3 PE3: bo[:4]=[212371 232194 174131  78223] nz=18816
L3 G3 skip=0: 0/75264 (0.00%)
L3 G3 skip=2: 56304/75264 (74.81%)


In [21]:

# ============================================================
# 셀 20: L4 (24→144→32, length 56→28, stride 2)
# ============================================================
run_layer(4, 24, 144, 32, 56, 2, 'L3_G3')



=== L4: in=24@56², mid=144, out=32, stride_g2=2, exp=1 ===
  restored L3_G3 (75264 ints)

--- L4 G1 (1x1 expand) ---
  G1 PE0: bo[:4]=[     0 105163 638216  62885] nz=64825
  G1 PE1: bo[:4]=[ 193023   49554 1557690  906681] nz=74909
  G1 PE2: bo[:4]=[     0      0 772676 622489] nz=63497
  G1 PE3: bo[:4]=[28711     0 76615 54478] nz=80683
    skip=0: 22.86%
    skip=2: 99.77%
  L4 G1: skip=2, 450560/451584 (99.77%)

--- L4 G2 (3x3 depthwise) ---
  G2 PE0: bo[:4]=[382038 134659 290646 311374] nz=22631
  G2 PE1: bo[:4]=[ 19370 217627 174637 150970] nz=23156
  G2 PE2: bo[:4]=[ 17764 317632 306649 260749] nz=21003
  G2 PE3: bo[:4]=[ 34000 111561      0      0] nz=22268
    skip=0: 13.45%
    skip=2: 13.20%
  L4 G2: skip=0, 15182/112896 (13.45%)

--- L4 G3 (1x1 project) ---
  G3 PE0: bo[:4]=[251144 -11610  37858 183632] nz=6272
  G3 PE1: bo[:4]=[-340492 -292326   82267  231586] nz=6272
  G3 PE2: bo[:4]=[-40122 157110 553059 205696] nz=6272
  G3 PE3: bo[:4]=[-806625 -289130 -543074 -272228]

In [22]:

# ============================================================
# 셀 21: L5 (32→192→32, length 28, stride 1)
# ============================================================
run_layer(5, 32, 192, 32, 28, 1, 'L4_G3')




=== L5: in=32@28², mid=192, out=32, stride_g2=1, exp=0 ===
  restored L4_G3 (25088 ints)

--- L5 G1 (1x1 expand) ---
  G1 PE0: bo[:4]=[495087  95175 378765 599928] nz=28703
  G1 PE1: bo[:4]=[1093791 1097799 1776815 1231262] nz=27050
  G1 PE2: bo[:4]=[     0      0 502538      0] nz=29791
  G1 PE3: bo[:4]=[1187303  825906  474752  418161] nz=29940
    skip=0: 17.15%
    skip=2: 99.76%
  L5 G1: skip=2, 150171/150528 (99.76%)

--- L5 G2 (3x3 depthwise) ---
  G2 PE0: bo[:4]=[     0 160010  28833      0] nz=20885
  G2 PE1: bo[:4]=[     0      0      0 239359] nz=19460
  G2 PE2: bo[:4]=[28672  7224  9314  9646] nz=19720
  G2 PE3: bo[:4]=[  6144  53624 212870  91082] nz=17215
    skip=0: 36.11%
    skip=2: 92.85%
  L5 G2: skip=2, 139764/150528 (92.85%)

--- L5 G3 (1x1 project) ---
  G3 PE0: bo[:4]=[0 0 0 0] nz=0
  G3 PE1: bo[:4]=[0 0 0 0] nz=0
  G3 PE2: bo[:4]=[0 0 0 0] nz=0
  G3 PE3: bo[:4]=[0 0 0 0] nz=0
    skip=0: 0.00%
    skip=2: 0.00%
  L5 G3: skip=0, 0/25088 (0.00%)


In [24]:

  import time, numpy as np
  from pynq import allocate

  # === L5 G3 with residual ADD ===

  # input 준비 (G2 ans로 복구한 cpu_map에서)
  ans_l5g2 = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L5_G2.txt', dtype=np.int32)
  buf_cpu_map[:len(ans_l5g2)] = ans_l5g2; buf_cpu_map.flush()

  p = d2s_1x1_exp0(buf_cpu_map, 32, 192, 28, 1)
  bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()

  # residual data (L4 G3 ans, PE 별로 분할)
  ans_l4g3 = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L4_G3.txt', dtype=np.int32)
  chunk = 32 * 28 * 28 // 4   # 6272 per PE
  res_bufs = []
  for pe in range(4):
      rd = allocate((chunk + 100,), dtype=np.int32); rd[:] = 0
      rd[:chunk] = ans_l4g3[pe*chunk:(pe+1)*chunk]
      rd.flush()
      res_bufs.append(rd)

  # 4 PE 호출
  g3 = []
  for pe in range(4):
      reset_all_dma()
      bo = allocate((10000,), dtype=np.int32); bo[:] = 0; bo.flush()
      rrecv = allocate((10000,), dtype=np.int32); rrecv[:] = 0; rrecv.flush()

      tp = buf_tile_convs.physical_address + tile_convs_offset_bytes(5, 2, pe)
      ip_phys = buf_info_convs.physical_address + info_convs_offset_bytes(5, 2, pe)
      ip_set_args(5, 2, 1)
      ip_set_tile_info(tp, ip_phys)

      res_wr.transfer(rrecv); ip_out.transfer(bo)
      res_rd.transfer(res_bufs[pe])    # ⭐ PE별 residual
      ip_in.transfer(bi)
      ctrl.write(0, 0x1)

      for _ in range(40):
          time.sleep(0.5)
          if (ctrl.read(0)>>1)&1: break
      nz = np.count_nonzero(np.asarray(bo))
      print(f"  G3 PE{pe}: bo[:4]={np.asarray(bo)[:4]} nz={nz}")
      g3.append(bo)

  # 검증
  ans = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L5_G3.txt', dtype=np.int32)
  for skip in [0, 2]:
      buf_cpu_map[:32*28*28] = 0; buf_cpu_map.flush()
      s2d_1x1_exp0(buf_cpu_map, g3, 32, 28, skip=skip)
      buf_cpu_map.flush()
      m = (ans == np.asarray(buf_cpu_map)[:len(ans)]).sum()
      print(f"L5 G3 skip={skip}: {m}/{len(ans)} ({100*m/len(ans):.2f}%)")


KeyboardInterrupt: 

In [ ]:

# ============================================================
# 셀 22: L6 (32→192→32, length 28, stride 1)
# ============================================================
run_layer(6, 32, 192, 32, 28, 1, 'L5_G3')


In [ ]:

# ============================================================
# 셀 23: L7 (32→192→64, length 28→14, stride 2)
# ============================================================
run_layer(7, 32, 192, 64, 28, 2, 'L6_G3')



In [ ]:

# ============================================================
# 셀 24: L8~L10 (64→384→64, length 14, stride 1)
# ============================================================
run_layer(8,  64, 384, 64, 14, 1, 'L7_G3')



In [ ]:

# In[25]:


run_layer(9,  64, 384, 64, 14, 1, 'L8_G3')



In [ ]:

# In[26]:


run_layer(10, 64, 384, 64, 14, 1, 'L9_G3')



In [ ]:

# In[27]:


# ============================================================
# 셀 27: L11 (64→384→96, length 14, stride 1)
# ============================================================
run_layer(11, 64, 384, 96, 14, 1, 'L10_G3')



In [ ]:

# In[28]:


# 셀 28: L12, L13 (96→576→96, length 14, stride 1)
run_layer(12, 96, 576, 96, 14, 1, 'L11_G3')



In [ ]:

# In[29]:


run_layer(13, 96, 576, 96, 14, 1, 'L12_G3')



In [ ]:

# In[30]:


# ============================================================
# 셀 30: L14 (96→576→160, length 14→7, stride 2)
# ============================================================
run_layer(14, 96, 576, 160, 14, 2, 'L13_G3')



In [ ]:

# In[31]:


# 셀 31: L15, L16 (160→960→160, length 7, stride 1)
run_layer(15, 160, 960, 160, 7, 1, 'L14_G3')



In [ ]:

# In[32]:


run_layer(16, 160, 960, 160, 7, 1, 'L15_G3')



In [ ]:

# In[33]:


# ============================================================
# 셀 33: L17 (160→960→320, length 7, stride 1)
# ============================================================
run_layer(17, 160, 960, 320, 7, 1, 'L16_G3')



In [ ]:

# In[ ]:


# ============================================================
# 셀 34: Final CONV + AVG + FC (별도 구현 필요)
# ============================================================
# TODO: 위 layer들이 100% 매치되면 추가
# - Final 1x1 CONV (320 → 1280)
# - AVG pooling (1280 × 7×7 → 1280)
# - FC (1280 → 1000) 또는 plant disease 분류 layer
print("L18, AVG, FC, classification 구현 필요")
